In [7]:
import sys
import os

# Get the absolute path of the parent directory
parent_dir = os.path.abspath('..')

# Add it to the system path if it isn't already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt


# 1. Fetch the dataset
california = fetch_california_housing(as_frame=True)

# 2. Extract MedInc as the single input feature (X)
# We keep it as a DataFrame (using double brackets) so it retains its column name and 2D shape
X = california.frame[["MedInc"]]

# 3. Extract the median house value as the target (y)
y = california.target

# Optional: Quick check to see what the data looks like
print("Feature (X) preview:")
print(X.head())
print("\nTarget (y) preview:")
print(y.head())

Feature (X) preview:
   MedInc
0  8.3252
1  8.3014
2  7.2574
3  5.6431
4  3.8462

Target (y) preview:
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64


In [2]:
california

{'data':        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
 0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
 1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
 2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
 3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
 4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
 ...       ...       ...       ...        ...         ...       ...       ...   
 20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
 20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
 20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
 20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
 20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981     39.37   
 
        Longitude 

In [3]:
# 1. Extract MedInc column as a 2D DataFrame for X
X = california['data'][['MedInc']]

# 2. Extract the target values for y
y = california['target']

# 3. Print to verify they loaded perfectly
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20640, 1)
y shape: (20640,)


In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    train_size=100, 
    test_size=100, 
    random_state=42
)

# Convert features to 1D numpy arrays to make the polynomial math much easier
x_train_arr = X_train["MedInc"].values
x_val_arr = X_val["MedInc"].values

# Convert targets to numpy arrays as well
y_train_arr = y_train.values
y_val_arr = y_val.values

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from src.linear_regression import LinearRegression

# Lists to store your MSE results for plotting later
train_mse_list = []
val_mse_list = []

degrees = range(1, 13) # d in {1, 2, ..., 12}

for d in degrees:
    # --- Feature Engineering ---
    # Construct the polynomial design matrix using list comprehension
    # This creates a matrix of [x^1, x^2, ..., x^d]
    X_train_poly = np.column_stack([x_train_arr**i for i in range(1, d + 1)])
    X_val_poly = np.column_stack([x_val_arr**i for i in range(1, d + 1)])
    
    # --- Standardization ---
    # Subtract mean and divide by std deviation to handle massive numbers
    scaler = StandardScaler()
    
    # FIT the scaler ONLY on the training data to avoid data leakage, 
    # then transform both training and validation sets.
    X_train_scaled = scaler.fit_transform(X_train_poly)
    X_val_scaled = scaler.transform(X_val_poly)
    
    # --- Fitting ---
    # Initialize YOUR custom model (replace with your actual class name if different)
    model = LinearRegression() 
    model.fit(X_train_scaled, y_train_arr)
    
    # --- Evaluation ---
    # Predict on both sets
    y_train_pred = model.predict(X_train_scaled)
    y_val_pred = model.predict(X_val_scaled)
    
    # Calculate Mean Squared Error
    mse_train = mean_squared_error(y_train_arr, y_train_pred)
    mse_val = mean_squared_error(y_val_arr, y_val_pred)
    
    # Store the results
    train_mse_list.append(mse_train)
    val_mse_list.append(mse_val)

print("Experiment loop complete! MSE values successfully stored.")

Experiment loop complete! MSE values successfully stored.


In [6]:
print(train_mse_list)
print(val_mse_list)

[0.6425542440376446, 0.6125081664317755, 0.6121250577105978, 0.599346675646922, 0.5967329316683174, 0.586329632678207, 0.5757119029711526, 0.5514246815998346, 0.5508030469534086, 0.5507709108022144, 0.5489867541277578, 0.5480642163770535]
[0.7662958469219403, 0.7624425654751513, 0.7663375653691109, 0.7572854560145978, 0.7629947414995022, 0.7768353128137748, 0.7543216502884195, 0.7456596421358168, 0.7500472315928005, 0.7502692955660464, 0.7516150447921077, 0.7508676878857632]



plt.figure(figsize=(10, 6))

# Plot the lines
plt.plot(degrees, train_mse_list, label='Training MSE', marker='o', color='blue')
plt.plot(degrees, val_mse_list, label='Validation MSE', marker='s', color='orange')

# Add required axis labels (crucial for your rubric!)
plt.xlabel('Degree of Polynomial (d)', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.title('Training and Validation MSE vs. Polynomial Degree', fontsize=14)

# Formatting to make it look clean
plt.xticks(degrees) # Forces the x-axis to show every integer from 1 to 12
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Display the plot
plt.show()

# --- 2. Report the degree with the lowest validation MSE ---
# Find the index of the minimum value in the validation MSE list
best_index = np.argmin(val_mse_list)
best_d = degrees[best_index]
lowest_val_mse = val_mse_list[best_index]

print(f"--- Results ---")
print(f"The degree with the lowest validation MSE is: d = {best_d}")
print(f"The lowest validation MSE achieved is: {lowest_val_mse:.4f}")